# 🚀 02 — Training YOLOv11n
**Tujuan:** Training YOLOv11n pada dataset Kaggle RFT (8 kelas sampah sungai).

> ✅ **Status:** Training 100 epoch sudah selesai. `weights/best.pt` sudah tersedia.
> Notebook ini untuk keperluan dokumentasi dan jika ingin melakukan training ulang.
>
> ⚠️ Training dari awal membutuhkan beberapa jam. Disarankan menggunakan GPU.


In [ ]:
import sys
sys.path.insert(0, '..')

import os
from pathlib import Path
import yaml
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'MPS available  : {torch.backends.mps.is_available()}')

# Check ultralytics
try:
    import ultralytics
    print(f'Ultralytics    : {ultralytics.__version__}')
except ImportError:
    print('❌ ultralytics not installed — pip install ultralytics')


## Step 1: Cek Model yang Sudah Ada

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch

weights_path = Path('../weights/best.pt')
print(f'Model tersedia: {weights_path.exists()}')
print(f'Ukuran file  : {weights_path.stat().st_size / 1024 / 1024:.2f} MB' if weights_path.exists() else '')

if weights_path.exists():
    model = YOLO(str(weights_path))
    print(f'✅ Model berhasil dimuat!')
    print(f'   Device: CPU')
    print(f'   Dataset: Kaggle RFT (8 kelas)')
    print(f'   Training: 100 epoch')
else:
    print('⚠️  weights/best.pt tidak ditemukan — lanjutkan ke training baru di bawah')


## Step 2: Cek Dataset

In [ ]:
dataset_dir = Path('../data/datasets')
print(f'Dataset dir: {dataset_dir}')
print(f'Exists     : {dataset_dir.exists()}')

for split in ['train', 'valid', 'test']:
    n = len(list((dataset_dir / split / 'images').glob('*'))) if (dataset_dir / split / 'images').exists() else 0
    print(f'  {split:6s}: {n} images')


## Step 2: Load Model

In [ ]:
from ultralytics import YOLO

# Try YOLOv11n first, fallback to YOLOv8n
model_candidates = ['yolo11n.pt', 'yolov8n.pt']
model = None
model_name_used = None

for candidate in model_candidates:
    try:
        model = YOLO(candidate)
        model_name_used = candidate
        print(f'✅ Loaded: {candidate}')
        break
    except Exception as e:
        print(f'⚠️  {candidate}: {e}')

if model is None:
    raise RuntimeError('Could not load any YOLO model')

# Model info
print(f'\nModel: {model_name_used}')
print(f'Parameters: {sum(p.numel() for p in model.model.parameters()):,}')


## Step 3: Konfigurasi Training Ulang (Opsional)

In [ ]:
# Konfigurasi training yang SUDAH DIGUNAKAN
# (Jalankan ini hanya jika ingin training ulang dari awal)
TRAIN_CONFIG = {
    'data': '../data/datasets/data.yaml',  # Dataset Kaggle RFT
    'epochs': 100,
    'imgsz': 640,
    'batch': 8,
    'device': '0' if torch.cuda.is_available() else 'cpu',
    'project': 'C:/yolo_out',
    'name': 'rft_run',
    'patience': 20,
    'lr0': 0.01,
    'lrf': 0.001,
    'cos_lr': True,
    'amp': False,
    'save': True,
}

print('Konfigurasi training:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')


## Step 4: Jalankan Quick Validation

In [ ]:
# Validasi model yang sudah ada pada test set
best_model = YOLO('../weights/best.pt')

import yaml
data_yaml_path = Path('../data/datasets/data.yaml')

val_results = best_model.val(
    data=str(data_yaml_path.resolve()) if data_yaml_path.exists() else None,
    split='test',
    imgsz=640,
    conf=0.15,
    iou=0.5,
    device='cpu',
    verbose=True
)

print('\n📊 Test Set Results:')
print(f'  mAP@0.5     : {val_results.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {val_results.box.map:.4f}')
print(f'  Precision   : {val_results.box.mp:.4f}')
print(f'  Recall      : {val_results.box.mr:.4f}')


## Step 5: Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# Cek hasil training curves dari Ultralytics
run_dir = Path('C:/yolo_out/rft_run')
results_csv = run_dir / 'results.csv'

if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = df.index + 1
    if 'metrics/mAP50(B)' in df.columns:
        axes[0].plot(epochs, df['metrics/mAP50(B)'], label='mAP@0.5', color='blue')
    if 'metrics/mAP50-95(B)' in df.columns:
        axes[0].plot(epochs, df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='orange')
    axes[0].set_title(f'Training Progress — YOLOv11n on Kaggle RFT')
    axes[0].set_xlabel('Epoch'); axes[0].legend()
    if 'train/box_loss' in df.columns:
        axes[1].plot(epochs, df['train/box_loss'], label='Box Loss', color='red')
    axes[1].set_title('Training Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout()
    plt.savefig('../results/visualizations/training_curves_rft.png', dpi=150)
    plt.show()
else:
    print(f'results.csv tidak ditemukan di: {run_dir}')


---

✅ **Training complete!**

Next step: Run `03_inference_evaluation.ipynb` for SAHI inference benchmark and comparison.